# Use `%%tensorguard` inside a notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/11_jupyter_magic.ipynb)

Load the TensorGuard IPython extension and verify a model cell in place. The magic checks the cell source before executing it, so a notebook experiment can surface tensor-contract bugs exactly where the model is written.

In [ ]:
%pip install -q "git+https://github.com/thehalleyyoung/tensorguard.git"

In [ ]:
%load_ext src.jupyter_integration

The next cell defines a broken attention-style block: the projection from 16 hidden units is followed by a head that incorrectly expects 12. `%%tensorguard` checks the cell with a symbolic batch shape before the class is left behind in the notebook namespace.

In [ ]:
%%tensorguard x=batch,8
import torch
import torch.nn as nn

class NotebookBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(8, 16)
        self.head = nn.Linear(12, 4)

    def forward(self, x):
        return self.head(torch.relu(self.embed(x)))

The pure helper is asserted below so CI proves the same model cell really produces a TensorGuard finding when the notebook is executed.

In [ ]:
from src.jupyter_integration import check_cell, format_cell_report

cell = '''
import torch
import torch.nn as nn
class NotebookBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(8, 16)
        self.head = nn.Linear(12, 4)
    def forward(self, x):
        return self.head(torch.relu(self.embed(x)))
'''
outcome = check_cell(cell, input_shapes={'x': ('batch', 8)})
print(format_cell_report(outcome))
assert outcome.checked
assert not outcome.safe
assert outcome.bug_count >= 1
assert 'NotebookBlock' in outcome.headline